In [1]:
import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import matplotlib.pyplot as plt # import matplotlib to visualize our qc metrics
import subprocess
import sys
import seaborn as sns
import numpy as np
from scipy.sparse import csr_matrix
import scanpy.external as sce
from sklearn.metrics import silhouette_score
import datetime
import numpy as np
import harmonypy as hm
import scanorama
import gffutils
from collections import defaultdict
import scipy.sparse as sp

METADATA_PATH_TMS = "/gpfs/commons/projects/knowles_singlecell_splicing/TabulaSenis/data/AWS/metadata/tabula-muris-senis-full-metadata.csv"
METADATA_PATH_AB = "/gpfs/commons/projects/knowles_singlecell_splicing/allen-brain/mouse_isocortex_hippocampal_2021/METADATA/metadata.csv"

# read the files in 
metadata_tms = pd.read_csv(METADATA_PATH_TMS, low_memory=False)
metadata_ab = pd.read_csv(METADATA_PATH_AB, low_memory=False)

In [2]:
def format_cell_id(index, group):
    """Format cell ID based on age group"""
    if group == '3m':
        parts = index.replace('.', '-', 1).replace('_', '-', 1).split('.')
        corrected_part = parts[1].replace('-', '_', 1)
        return parts[0] + '-' + corrected_part + '-1-1'
    return index.split('.')[0]

# metacolumn in TMS is meta_cell_column="cell_id"
metadata_tms_subset = metadata_tms[metadata_tms['method'] == 'facs'].copy()
metadata_tms_subset['cell_id'] = metadata_tms_subset.apply(
    lambda row: format_cell_id(row['index'], row['age']), axis=1)
metadata_tms_subset = metadata_tms_subset[['cell_id', 'age', 'cell_ontology_class', 
                                 'mouse.id', 'sex', 'subtissue', 'tissue']]

metadata_tms_subset.head()

,cell_id,age,cell_ontology_class,mouse.id,sex,subtissue,tissue
245389,A10_B000497_B009023_S10,18m,bulge keratinocyte,18_53_M,male,NaN,Skin
245390,A10_B000756_B007446_S10,18m,pancreatic B cell,18_45_M,male,Endocrine,Pancreas
245391,A10_B000802_B009022_S10,18m,bulge keratinocyte,18_47_F,female,Skin Anagen,Skin
245392,A10_B000927_B007456_S10,18m,skeletal muscle satellite cell,18_46_F,female,Muscle Diaphragm,Limb_Muscle
245393,A10_B001361_B007505_S10,18m,B cell,18_46_F,female,RV,Heart


In [3]:
sample_group = pd.read_csv("/gpfs/commons/projects/knowles_singlecell_splicing/allen-brain/mouse_isocortex_hippocampal_2021/METADATA/GSE185862_sample_group_mapping.csv.gz")
sra_AB = pd.read_csv("/gpfs/commons/projects/knowles_singlecell_splicing/allen-brain/mouse_isocortex_hippocampal_2021/SRA/SraRunTable.txt")
GSE_metadata = pd.read_csv("/gpfs/commons/projects/knowles_singlecell_splicing/allen-brain/mouse_isocortex_hippocampal_2021/METADATA/GSE185862_metadata_ssv4.csv.gz", low_memory=False)

In [4]:
# converting SRA to cell IDs 
sra_cell_ids = pd.read_csv("/gpfs/commons/projects/knowles_singlecell_splicing/allen-brain/mouse_isocortex_hippocampal_2021/sra_s3_links.txt", sep="\t", header=None)
# rename columns SRR and "expected_R1_fastq" 
sra_cell_ids.columns = ['SRR', 'expected_R1_fastq']

# Step 1: Extract the filename from the S3 path
sra_cell_ids["filename"] = sra_cell_ids["expected_R1_fastq"].str.extract(r"/([^/]+)$")[0]

# Step 2: Clean suffix like `.fastq.gz.1` → `.fastq.gz`
sra_cell_ids["expected_R1_fastq"] = sra_cell_ids["filename"].str.replace(r"\.fastq.*", ".fastq.gz", regex=True)

# Preview result
print(sra_cell_ids.head())

           SRR                             expected_R1_fastq  \
0  SRR16439216  LS-15348_E1-50_GGACTCCT-CTAAGCCT_R1.fastq.gz   
1  SRR16439217  LS-15348_E1-50_GGACTCCT-CGTCTAAT_R1.fastq.gz   
2  SRR16439218  LS-15007_E1-50_TAGGCATG-GTAAGGAG_R1.fastq.gz   
3  SRR16439219  LS-15348_E1-50_GGACTCCT-TCTCTCCG_R1.fastq.gz   
4  SRR16439221  LS-15348_E1-50_TAGGCATG-TATCCTCT_R1.fastq.gz   

                                         filename  
0     LS-15348_E1-50_GGACTCCT-CTAAGCCT_R1.fastq.1  
1     LS-15348_E1-50_GGACTCCT-CGTCTAAT_R1.fastq.1  
2  LS-15007_E1-50_TAGGCATG-GTAAGGAG_R1.fastq.gz.1  
3     LS-15348_E1-50_GGACTCCT-TCTCTCCG_R1.fastq.1  
4     LS-15348_E1-50_TAGGCATG-TATCCTCT_R1.fastq.1  


In [5]:
GSE_metadata = GSE_metadata.merge(sra_cell_ids, on="expected_R1_fastq")
GSE_metadata.head()

,Unnamed: 0,sample_name,donor_sex_id,donor_sex_label,donor_sex_color,region_id,region_label,region_color,platform_label,cluster_order,...,cell_type_designation_color,cell_type_alt_alias_color,cell_type_alias_color,cell_type_accession_color,cortical_layer_label,cortical_layer_order,cortical_layer_color,expected_R1_fastq,SRR,filename
0,1,US-1250273_E1_S37,1,F,#565353,1,VISp,#9299FF,SS,259,...,#286291,#286291,#286291,#286291,L4/5/6,11,#7373FF,US-1250273_E1_GGACTCCT-GTAAGGAG_R1.fastq.gz,SRR16454043,US-1250273_E1_GGACTCCT-GTAAGGAG_R1.fastq.1
1,2,US-1250273_E2_S01,2,M,#ADC4C3,1,VISp,#9299FF,SS,259,...,#286291,#286291,#286291,#286291,L4/5/6,11,#7373FF,US-1250273_E2_TAAGGCGA-GCGTAAGA_R1.fastq.gz,SRR16454044,US-1250273_E2_TAAGGCGA-GCGTAAGA_R1.fastq.gz.1
2,3,US-1250273_E2_S02,2,M,#ADC4C3,1,VISp,#9299FF,SS,259,...,#286291,#286291,#286291,#286291,L4/5/6,11,#7373FF,US-1250273_E2_TAAGGCGA-CTCTCTAT_R1.fastq.gz,SRR16454045,US-1250273_E2_TAAGGCGA-CTCTCTAT_R1.fastq.gz.1
3,4,US-1250273_E2_S03,2,M,#ADC4C3,1,VISp,#9299FF,SS,197,...,#4EA8AC,#4EA8AC,#4EA8AC,#4EA8AC,L4/5/6,11,#7373FF,US-1250273_E2_TAAGGCGA-TATCCTCT_R1.fastq.gz,SRR16454046,US-1250273_E2_TAAGGCGA-TATCCTCT_R1.fastq.gz.1
4,5,US-1250273_E2_S04,2,M,#ADC4C3,1,VISp,#9299FF,SS,245,...,#0D5D7E,#0D5D7E,#0D5D7E,#0D5D7E,L4/5/6,11,#7373FF,US-1250273_E2_TAAGGCGA-AGAGTAGA_R1.fastq.gz,SRR16454047,US-1250273_E2_TAAGGCGA-AGAGTAGA_R1.fastq.gz.1


### Load gene expression
This data set includes single-cell transcriptomes from multiple cortical areas and the hippocampal formation, including 77K total cells. Samples were collected from dissections of brain regions from ~8 week-old male and female mice, primarily from pan-GABAergic, pan-glutamatergic, and pan-neuronal transgenic lines, with the addition of more specific transgenic lines and some retrogradely-labeled cells in VISp and ALM.

In [6]:
# Set up paths for files 
gene_exp = "/gpfs/commons/projects/knowles_singlecell_splicing/allen-brain/mouse_isocortex_hippocampal_2021/GeneExpression/matrix.csv" #intronic and exonic reads 
introns_only = "/gpfs/commons/projects/knowles_singlecell_splicing/allen-brain/mouse_isocortex_hippocampal_2021/GeneExpression/matrix.csv/intron.csv"
exons_only = "/gpfs/commons/projects/knowles_singlecell_splicing/allen-brain/mouse_isocortex_hippocampal_2021/GeneExpression/matrix.csv/exon.csv"
read_introns_exons = False 

if read_introns_exons:
    ab_adata_introns = sc.read_csv(introns_only)
    ab_adata_introns = ab_adata_introns.transpose()
    print(f"Done reading {introns_only}")

    ab_adata_exons = sc.read_csv(exons_only)
    ab_adata_exons = ab_adata_exons.transpose()
    print(f"Done reading {exons_only}")
    
#ab_adata = pd.read_csv(gene_exp, index_col=0)
#print(f"Done reading {gene_exp}")

In [7]:
def preprocess_ab_adata(adata, metadata, dataset_label="allen_brain", 
                        metadata_key="sample_name", rename_var=True):
    """
    Standardizes Allen Brain adata object:
    - Subsets metadata to only matching cells
    - Renames gene info
    - Stores raw counts layer
    """
    adata = adata.copy()
    adata.var['gene_name'] = adata.var_names
    adata.obs["dataset"] = dataset_label

    # Use correct column to index metadata
    if metadata_key not in metadata.columns:
        raise ValueError(f"Metadata key '{metadata_key}' not found in metadata columns.")
    
    metadata_sub = metadata[metadata[metadata_key].isin(adata.obs_names)].copy()
    metadata_sub = metadata_sub.set_index(metadata_key)

    # Align metadata to adata
    adata = adata[adata.obs_names.isin(metadata_sub.index)].copy()
    adata.obs = metadata_sub.loc[adata.obs_names]

    # Rename gene_name -> gene_symbol
    if rename_var:
        adata.var.rename(columns={"gene_name": "gene_symbol"}, inplace=True)
    else:
        adata.var["gene_symbol"] = adata.var["gene_name"]
    adata.layers["raw_counts"] = adata.X.copy()
    return adata

# Process total counts normally (uses 'sample_name')
#ab_adata = preprocess_ab_adata(ab_adata, metadata_ab, metadata_key="sample_name")

# Process exon and intron counts (uses 'exp_component_name')
#if read_introns_exons:
#    ab_adata_exons = preprocess_ab_adata(ab_adata_exons, metadata_ab, metadata_key="exp_component_name")
#    ab_adata_introns = preprocess_ab_adata(ab_adata_introns, metadata_ab, metadata_key="exp_component_name")

#print("✅ All AB datasets preprocessed with correct metadata mapping.")
#print(ab_adata)

In [8]:
GSE_metadata["Run"] = GSE_metadata["SRR"]

In [9]:
sra_AB = sra_AB.merge(GSE_metadata, on="Run")

In [10]:
sra_AB.iloc[0]["sample_name"]

'LS-15348_S38_E1-50'

In [11]:
sra_AB.cell_type_alias_label.value_counts().to_string()

'cell_type_alias_label\n168_L2/3 IT CTX            3967\n290_L6 CT CTX              3277\n180_L4 IT CTX              2236\n179_L4 IT CTX              2040\n12_Lamp5                   1800\n363_DG                     1780\n191_L4/5 IT CTX            1710\n227_L6 IT CTX              1425\n228_L6 IT CTX              1228\n51_Vip                     1050\n265_L5/6 NP CTX            1038\n82_Sst                      999\n114_Pvalb                   919\n376_Astro                   907\n182_L4/5 IT CTX             862\n11_Lamp5                    843\n188_L4/5 IT CTX             834\n116_Pvalb                   808\n44_Vip                      766\n238_Car3                    703\n288_L6 CT CTX               614\n46_Vip                      601\n189_L4/5 IT CTX             533\n197_L5 IT CTX               529\n98_Sst                      529\n304_L6b CTX                 512\n187_L4/5 IT CTX             510\n43_Vip                      507\n266_L5/6 NP CTX             506\n183_L4/5 IT CTX    

In [12]:
# print all values in the first row of sra_AB
print(sra_AB.iloc[0].to_string())

Run                                                             SRR16439216
Assay Type                                                          RNA-Seq
AvgSpotLen                                                              102
Bases                                                             263115834
BioProject                                                      PRJNA772116
BioSample                                                      SAMN22366283
Bytes                                                             132028277
Center Name                                                             GEO
Consent                                                              public
DATASTORE filetype                                         fastq,run.zq,sra
DATASTORE provider                                               gs,ncbi,s3
DATASTORE region                       gs.us-east1,ncbi.public,s3.us-east-1
Experiment                                                      SRX12667398
GEO_Accessio

In [13]:
sra_AB["cell_id"] = sra_AB["Run"]
sra_AB["cell_ontology_class"] = sra_AB["cell_type_alias_label"]
sra_AB["tissue"] = sra_AB["sample_name"]
sra_AB["age"] = "2m" 
sra_AB["mouse.id"] = sra_AB["external_donor_name_label"] # not sure if this is actually a mouse ID but most likely? 
sra_AB["subtissue"] = sra_AB["region_label"] 
sra_AB["sex"] = sra_AB["donor_sex_label"]

# add columns 
sra_AB_subset = sra_AB[["cell_id", "age", "cell_ontology_class", "mouse.id", "sex", "subtissue", "tissue"]]
sra_AB_subset

,cell_id,age,cell_ontology_class,mouse.id,sex,subtissue,tissue
0,SRR16439216,2m,287_L6 CT CTX,245824,M,VISp,LS-15348_S38_E1-50
1,SRR16439217,2m,227_L6 IT CTX,245824,M,VISp,LS-15348_S39_E1-50
2,SRR16439218,2m,227_L6 IT CTX,225974,M,VISp,LS-15007_S43_E1-50
3,SRR16439219,2m,227_L6 IT CTX,245824,M,VISp,LS-15348_S40_E1-50
4,SRR16439221,2m,228_L6 IT CTX,245824,M,VISp,LS-15348_S42_E1-50
...,...,...,...,...,...,...,...
69302,SRR16459076,2m,197_L5 IT CTX,366890,F,VISp,SM-GE676_S087_E1-50
69303,SRR16459078,2m,197_L5 IT CTX,366890,F,VISp,SM-GE676_S089_E1-50
69304,SRR16459085,2m,257_L5 PT CTX,366890,F,VISp,SM-GE676_S095_E1-50
69305,SRR16459086,2m,197_L5 IT CTX,366890,F,VISp,SM-GE676_S096_E1-50


In [14]:
# combine the two dataframes
metadata_combined = pd.concat([metadata_tms_subset, sra_AB_subset], ignore_index=True)

In [15]:
metadata_combined.age.value_counts()

age
2m     69307
3m     44518
18m    34027
24m    31551
21m      728
Name: count, dtype: int64

In [16]:
# save in /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION/
metadata_combined.to_csv("/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION/metadata_mouse_metadata_combined.csv", index=False)